In [3]:
from ukrdc.database import Connection
from sqlalchemy.orm import sessionmaker

engine = Connection.get_engine_from_file(key="ukrdc_staging")

ukrdc3_sessionmaker = sessionmaker(
    autocommit=False, autoflush=False, bind=engine
)

ukrdc3 = ukrdc3_sessionmaker()



# Patients Treatment Statistics
The following shows how to generate some statistics demonstrating the propotion of patients on home therapies. These are calculated from June 2021 to June 2022 and show the breakdown of the different dialysis types as well as the treatment is occuring at home or in centre. 

The final histogram shows the mean number of times each in-centre dialysis patient recieves RRT. Statistics like this can be used to show the general compsition of the patients and the treatments they are receiving.  

In [13]:

from ukrdc_stats.calculators.dialysis import DialysisStatsCalculator
import datetime as dt

calculator = DialysisStatsCalculator(
    ukrdc3, "RNJ00", from_time=dt.datetime(2022, 12, 31), to_time=dt.datetime(2023, 12, 31)
)

dialysis_stats = calculator.extract_stats()



In [4]:
print(list(dialysis_stats.units.keys()))

['RNJ00', '9RNJ00', 'RGCNH', 'RGCKH', 'RF4DG', '8CJ07']


In [14]:
from os import access
from ukrdc_stats.calculators.dialysis import DialysisStatsCalculator
import plotly.graph_objects as go
import plotly.express as px

import datetime as dt
from IPython.display import display

print(list(dialysis_stats.units.keys()))
#subunit = "RGCKH"

print(dialysis_stats.all.incident_krt.metadata.population_size)

prev_patients = px.pie(
    names = dialysis_stats.all.incident_krt.data.x,
    values = dialysis_stats.all.incident_krt.data.y,
    title =  dialysis_stats.all.incident_krt.metadata.title,
    hole=0.3,
)
prev_patients.show()



#print(dialysis_stats.all.incident_home_therapies.metadata.population_size)
incident_patients = px.pie(
    names = dialysis_stats.all.prevalent_krt.data.x,
    values = dialysis_stats.all.prevalent_krt.data.y,
    title =  dialysis_stats.all.prevalent_krt.metadata.title,
    hole=0.3,
)
incident_patients.show()


['RNJ00', 'RGCKH', '9RNJ00', 'RGCNH', 'RF4DG', '8CJ07']
360


: 

In [5]:
freq_fig = px.bar(
    x=dialysis_stats.all.incentre_dialysis_frequency.data.y,
    y=dialysis_stats.all.incentre_dialysis_frequency.data.x,
    title=dialysis_stats.all.incentre_dialysis_frequency.metadata.title,
    labels={
        "x": dialysis_stats.all.incentre_dialysis_frequency.metadata.axis_titles.y,
        "y": dialysis_stats.all.incentre_dialysis_frequency.metadata.axis_titles.x,
    },
    orientation="h",
    color_discrete_sequence=["green"],
    text_auto=True,
)

# set white background
freq_fig.update_layout({"plot_bgcolor": "rgba(0,0,0,0)"})
freq_fig.show()

access_fig = px.pie(
    values=dialysis_stats.all.incident_initial_access.data.y,
    names=dialysis_stats.all.incident_initial_access.data.x,
    title=dialysis_stats.all.incident_initial_access.metadata.title,
    hole=0.3,
)
access_fig.show()

# 